In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window
import pyspark.sql.functions as F

In [0]:
bronze_table = "learning_lab_catalog.bronze.fact_transaction"
silver_table = "learning_lab_catalog.silver.fact_transaction"

In [0]:
source_df = spark.table(bronze_table)

In [0]:
source_df_fixed = (
    source_df
    .withColumn(
        "updated_at",
        F.coalesce(
            F.expr("try_to_timestamp(updated_at, 'MMM dd yyyy  h:mma')"),
            F.expr("try_to_timestamp(updated_at, 'yyyy-MM-dd HH:mm:ss')"),
            F.expr("try_to_timestamp(updated_at)")
        )
    )
    .withColumn(
        "created_at",
        F.coalesce(
            F.expr("try_to_timestamp(created_at, 'MMM dd yyyy  h:mma')"),
            F.expr("try_to_timestamp(created_at, 'yyyy-MM-dd HH:mm:ss')"),
            F.expr("try_to_timestamp(created_at)")
        )
    )
)

window_spec = (
    Window
    .partitionBy("Transaction_ID")
    .orderBy(F.col("updated_at").desc_nulls_last())
)

deduped_source_df = (
    source_df_fixed
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)
if spark.catalog.tableExists(silver_table):

    target = DeltaTable.forName(spark, silver_table)

    (
        target.alias("t")
        .merge(
            deduped_source_df.alias("s"),
            "t.Transaction_ID = s.Transaction_ID"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(" Silver FACT incremental merge completed")

else:

    (
        deduped_source_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

    print("Silver FACT initial load completed")


In [0]:
%sql
select count(*) from learning_lab_catalog.silver.fact_transaction